# Read hums, get f0 and segment to notes.

Import all necessary packages and modules.

In [ ]:
from itertools import groupby

import librosa
import librosa.display
import matplotlib.pyplot as plt
import mir_eval
import numpy as np
import scipy
import soundfile
from glob import glob

%matplotlib inline

Load the hums as wav audio files.

In [ ]:
# Specify directory and local paths to an improv song of a participant.

datapath = "C:/Users/monik/Documents/Paris/Internship/Analyses/999/Improv/"
filename = "999_recording_mic_4.wav"
filepath = f'{datapath}{filename}'

# Read in the file.
y, sr = librosa.load(filepath)

## Get the fundamental frequencies (f0).

Perform the short-time Fourier transform (STFT) to analyze frequency changes over time.

In [ ]:
# Compute the spectrogram.
D = librosa.stft(y)

# Convert an amplitude spectrogram to dB-scaled spectrogram.
S_db = librosa.amplitude_to_db(np.abs(D), ref=np.max)

# Perform the fundamental frequency (F0) estimation with pYIN algorithm. The algorithm returns:
    # f0 = time series of fundamental frequencies in Hertz
    # vid = time series containing boolean flags indicating whether a frame is voiced or not.
    # vpd = time series containing the probability that a frame is voiced

f0, vid, vpd = librosa.pyin(
        y,
        sr=sr,
        fmin=librosa.note_to_hz("C1"),
        fmax=librosa.note_to_hz("C7"),
        resolution=0.1, # 0.01 corresponds to cent resolution but it takes longer, change if necessary for other analyses
    )

# Create a numpy array of time frames.
tf = np.linspace(start=0, stop =(len(y) / sr), num=len(f0))

# Plot the spectrogram: tf over f0.
fig, ax = plt.subplots(figsize=(10, 4))
img = librosa.display.specshow(S_db, x_axis="time", y_axis="log", ax=ax, sr=sr)
fig.colorbar(img, ax=ax, format="%2.f dB")
ax.plot(tf, f0)
ax.set(title=filename, ylim=[75, 475]) # setting the y-axis view limits

##  Get a chromagram to identify chroma at each time point.

In [ ]:
# optimize filtering
chroma = librosa.feature.chroma_cqt(y=y, sr=sr, norm=None) # In CQT chromagram
                                                           # (Constant-Q
                                                           # Tranform) one 
                                                           # vertical step
                                                           # represents one
                                                           # semitone
# sparse de-noising of the chromagram
chroma_filter = np.minimum(
    chroma, librosa.decompose.nn_filter(chroma, aggregate=np.median, metric="cosine")
)

kernel = np.tile(np.array([-0.5, 1, -0.5])[:, None], (1, 9)) # Calculate the absolute value element-wise.
kernel /= np.abs(kernel).sum() # kernel / np.abs(kernel).sum()

# Multidimensional convolution
chroma_smooth = scipy.ndimage.filters.convolve(chroma_filter, kernel)

# Threshold
chroma_thresh = chroma_smooth
chroma_thresh[chroma_thresh < 0.1] = -np.inf # values become a negative numpy infinity

# Plot
fig, ax = plt.subplots(nrows=2, sharex=True, figsize=(10, 4))
img = librosa.display.specshow(S_db, x_axis="time", y_axis="log", ax=ax[0], sr=sr)
fig.colorbar(img, ax=ax[0], format="%2.f dB")
tf = np.linspace(0, len(y) / sr, len(f0))
ax[0].plot(tf, f0)
ax[0].set(title=filename, ylim=[75, 475])

img = librosa.display.specshow(chroma_thresh, y_axis="chroma", x_axis="time", ax=ax[1]) # The y axis is pitch class ....
cb = fig.colorbar(img, ax=ax[1])

ax[1].set(title="Chroma With Filtering and Smoothing")
index = np.argmax(chroma_thresh, axis=0).astype(float)
nanid = np.all(np.isinf(chroma_thresh), axis=0)
index[nanid] = np.nan
inList = list()
for k, g in groupby(index):
    seg = [*g]
    if len(seg) > 3:
        inList += seg
    else:
        inList += [np.nan] * len(seg)
index = np.array(inList)

ax[1].plot(tf, index, ".")
ax[1].set_xlim([0, 14])

#fig.show()
#fig.savefig(f"f0vChroma_{filename.replace('.wav', '')}")

## Math octaves maintaining duration and compare transcription in wavefile

In [ ]:
index_labels = ["C", "C#", "D", "D#", "E", "F", "F#", "G", "G#", "A", "A#", "B"]
hz = list()
midi_notes = list()

for k, g in groupby(zip(index, f0), key=lambda x: x[0]):
    freqs = [t[1] for t in g]
    if np.isnan(k):
        hz.append(k)
        midi_notes.append(k)
    else:

        # Try to identify which note and which octave
        currlab = index_labels[int(k)]
        posF = librosa.note_to_hz([f"{currlab}{i}" for i in range(1, 7)])
        avF = np.nanmean(freqs)
        octI = np.argmin(np.abs(posF - avF)) + 1
        note_name = f"{currlab}{octI}"
        
        # Convert to Hz and Midi
        freq = librosa.note_to_hz(note_name)
        midi = librosa.note_to_midi(note_name)
        validSamp = ~np.isnan(freqs)
        hz += list(freq * validSamp)
        midi_notes += [midi if vS else np.nan for vS in validSamp]

# Put into numpy array
hz = np.array(hz)
hz[np.isnan(hz)] = -1

# Create acoustic time-series from array of notes (hz)
y_chroma = mir_eval.sonify.pitch_contour(tf, hz, sr, length=len(y))
rms_y = np.sqrt(np.mean(y ** 2))
rms_yC = np.sqrt(np.mean(y_chroma ** 2))
y_chroma *= 2 * rms_y / rms_yC

# Save chroma WAV
soundfile.write(filename.replace(".wav", "_y_chroma.wav"), y_chroma, sr)


# Put both original and chroma into same array and save as different channels of WAV file
side_by_side = np.vstack([y, y_chroma]).T
soundfile.write(filename.replace(".wav", "_transcript.wav"), side_by_side, sr)

## Plot transition probabilites from chroma to chroma in polar   

In [ ]:
fig, ax = plt.subplots(ncols=2, subplot_kw={"projection": "polar"})
fphase = 2 * np.pi * (f0 - 125) / 125

ax[0].plot(fphase, tf)
ax[0].set(title="raw F0")

 
ax[1].plot(cphase, tf)
ax[1].set(title="Chroma")

#fig.show()

sequence = np.array(
    [[k, len([*g])] for k, g in groupby(index[~np.isnan(index)])]
).astype(int)

trans = np.zeros([12, 12])

for i in range(1, len(sequence)):
    curr, prev = sequence[i, 0], sequence[i - 1, 0]
    trans[prev, curr] += 1

med_durations = np.zeros([12, 12])
for prev in range(12):
    for curr in range(12):
        # get median time length for each
        select = np.insert(
            (sequence[:-1, 0] == prev) & (sequence[1:, 0] == curr), 0, False
        )
        med_durations[prev, curr] = np.median(sequence[select, 1])

fig, ax = plt.subplots(ncols=2, subplot_kw={"projection": "polar"})
ax[0].plot(sequence[:, 0], np.ones(len(sequence)))
ax[0].set(title="Overlapping sequence")

cmap = plt.cm.viridis #RGBA values-based colormap
vmax = 30.0
for prev in range(12):
    pPh = 2 * np.pi * prev / 12
    for curr in range(12):
        cPh = 2 * np.pi * curr / 12
        thisTrans = trans[prev, curr]
        thisDur = med_durations[prev, curr]
        if thisTrans > 0:
            ax[1].plot(
                [pPh, cPh],
                [1, 1],
                lw=6 * thisTrans / trans.max(),
                color=cmap(np.minimum(thisDur / vmax, 1.0)),
            )
values, size = np.unique(sequence[:, 0], return_counts=True)

vPh = 2 * np.pi * values / 12
ax[1].scatter(vPh, np.ones_like(vPh), color="k", s=6 * size, zorder=100)

# Plot the sequence and the transition probabilities.
ax[1].set(title="Transition probabilities")
sm = plt.cm.ScalarMappable(plt.Normalize(0, vmax * 512 / sr), cmap)
plt.tight_layout()
cax = fig.add_subplot(10, 2, 20)
cb = fig.colorbar(sm, cax=cax, orientation="horizontal", ticklocation="top")
    
#fig.show()
#fig.savefig(f"transition_probabilities_polar_{filename.replace('.wav', '')}")

## Keep track of octaves to make relative pitch work well.

In [ ]:
index_labels = ["C", "C#", "D", "D#", "E", "F", "F#", "G", "G#", "A", "A#", "B"]
note_names = list()
for k, g in groupby(
    zip(index[~np.isnan(index)], f0[~np.isnan(index)]), key=lambda x: x[0]
):
    freq = [t[1] for t in g]
    nSamps = sum(~np.isnan(freq))
    if nSamps > 0:
        currlab = index_labels[int(k)]
        posF = librosa.note_to_hz([f"{currlab}{i}" for i in range(1, 7)])
        avF = np.nanmean(freq)
        octI = np.argmin(np.abs(posF - avF)) + 1
        note_names.append(f"{currlab}{octI}")
note_names = np.array(note_names)
midi_index = librosa.note_to_midi(note_names)
semitones = np.diff(midi_index)
fig = plt.figure()
#plt.hist(semitones, bins=np.arange(semitones.min() - 0.5, semitones.max() + 0.5, 1))
#fig.show()
#fig.savefig(f"octaves_track_{filename.replace('.wav', '')}")

## Plot transition probabilities.

In [ ]:
xy = range(np.min([-12, semitones.min()]), np.max([13, semitones.max() + 1]))
semitrans = np.zeros([len(xy)] * 2)
for i in range(1, len(semitones)):
    curr, prev = semitones[i], semitones[i - 1]
    semitrans[xy.index(prev), xy.index(curr)] += 1.0
fig, ax = plt.subplots()
semitrans[semitrans == 0] = np.nan
# semitrans /= np.nansum(semitrans, axis=1, keepdims=True)
semitrans /= np.nansum(semitrans)
img = ax.pcolormesh(xy, xy, semitrans, shading="nearest", cmap=plt.cm.cividis)
ax.set(xlabel="Current Interval", ylabel="Previous Interval")
fig.colorbar(img, ax=ax)
#fig.show()
#fig.savefig(f"transition_probabilities_{filename.replace('.wav', '')}")